# Topic modeling

Some of this code is partially adapted from [here](https://github.com/kapadias/medium-articles/blob/master/natural-language-processing/topic-modeling/Introduction%20to%20Topic%20Modeling.ipynb)

In [1]:
import gensim, tqdm, nltk, json
from zipfile import ZipFile
RANDOM_SEED = 15
stop_words = nltk.corpus.stopwords.words('english')

In [6]:
def document_reader_bigfile(bin, bigfilename="/home/users1/zabereus/zabereus/master_env/RSC603Full.vrt"):
    """ Generator for RSC sentences from selected time bin.
    """
    bin_ixs = [0, 3011, 4830, 7604, 14357, 24368, 47836]
    if bin == 0 or bin == 'all':
        start, end = 0, bin_ixs[-1]
    else:
        start, end = bin_ixs[bin-1], bin_ixs[bin]
    with open(bigfilename, 'r') as f:
        buffer = ""
        docs = 0
        for line in tqdm.tqdm(f):
            if '<' not in line:
                buffer += ' ' + line.split()[0] #remove all xml junk, pos etc
            if "</text>" in line:
                if docs > start:
                    #TODO exclude bad langs etc
                    if True:
                        yield buffer
                buffer = ""
                docs += 1
            if docs >= end:
                break

In [2]:
def document_reader_coha(decades=range(1810, 2010, 10)):
    """
    Generator yielding COHA documents as strings, tokens separated by spaces
    """
    path_to_corpus = "/mount/resources/corpora/COHA/CCOHA/tagged/"
    for decade in decades:
        with ZipFile(path_to_corpus + f'cleaned_{decade}s.zip') as zf:
            for file in zf.namelist():
                with zf.open(file) as f:
                    tokens = []
                    for line in f.readlines():
                        token = line.decode().split('\t')[0].strip()
                        if token != "<eos>":
                            tokens.append(token)
                        if len(tokens) > 5000:
                            yield ' '.join(tokens)  
                            tokens = []          
                yield ' '.join(tokens)

In [3]:
#documents = list(document_reader_bigfile(bin='all'))
documents = list(document_reader_coha())

def docs_to_words(documents):
    for doc in tqdm.tqdm(documents):
        # deacc=True removes punctuations
        yield [word for word in gensim.utils.simple_preprocess(doc, deacc=True) if word not in stop_words]

tokenized_docs = list(docs_to_words(documents))

print(tokenized_docs[0])


100%|██████████| 182791/182791 [18:22<00:00, 165.80it/s]

['persons', 'represented', 'men', 'pufpace', 'old', 'miser', 'charles', 'miser', 'son', 'meanel', 'honest', 'man', 'goodel', 'brother', 'law', 'pufpace', 'quicksite', 'lawyer', 'jake', 'servant', 'miser', 'stranger', 'hardy', 'sea', 'captain', 'harry', 'meanel', 'servant', 'women', 'eliza', 'meanel', 'daughter', 'molly', 'meanel', 'maid', 'main', 'text', 'act', 'scene', 'room', 'old', 'misers', 'house', 'jake', 'enters', 'whip', 'hand', 'enters', 'halloos', 'cattle', 'without', 'takes', 'bottle', 'cider', 'closet', 'drinks', 'hearing', 'noise', 'door', 'hides', 'haste', 'thinking', 'master', 'coming', 'enter', 'stranger', 'pack', 'stran', 'stranger', 'halloo', 'jake', 'jake', 'stran', 'stranger', 'lives', 'jake', 'jake', 'stran', 'stranger', 'owns', 'house', 'jake', 'jake', 'man', 'lives', 'stran', 'stranger', 'stran', 'stranger', 'lives', 'besides', 'jake', 'jake', 'peg', 'sip', 'jowler', 'stran', 'stranger', 'jowler', 'jake', 'jake', 'yes', 'master', 'dog', 'stran', 'stranger', 'nobo

In [4]:
#id2word = gensim.corpora.dictionary.Dictionary.load("coha/lda_coha/lda_coha.model.id2word")#("lda/lda.model.id2word")
#corpus = json.load(open("coha/lda_coha/lda_corpus_coha.json", "r"))#"lda/lda_corpus.json"
id2word = gensim.corpora.Dictionary(tokenized_docs)
id2word.filter_extremes(no_below=5, no_above=0.5, keep_n=2000)
corpus = [id2word.doc2bow(tokens) for tokens in tokenized_docs]
print(corpus[0][:30])

[(0, 3), (1, 1), (2, 1), (3, 1), (4, 1), (5, 2), (6, 14), (7, 1), (8, 3), (9, 1), (10, 1), (11, 2), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 3), (18, 1), (19, 1), (20, 2), (21, 1), (22, 2), (23, 1), (24, 1), (25, 1), (26, 3), (27, 1), (28, 1), (29, 1)]


In [ ]:
for i in range(2000):
    if id2word[i] in """(18, '0.007*"upon" + 0.006*"man" + 0.006*"old" + 0.006*"may" + 0.006*"like" + 0.006*"well" + 0.006*"mr" + 0.005*"know" + 0.005*"little" + 0.005*"much"')
(2, '0.007*"upon" + 0.007*"mr" + 0.006*"like" + 0.005*"may" + 0.005*"good" + 0.005*"man" + 0.005*"little" + 0.005*"well" + 0.005*"great" + 0.005*"must"')
(6, '0.007*"man" + 0.006*"upon" + 0.006*"mr" + 0.006*"may" + 0.006*"like" + 0.005*"us" + 0.005*"every" + 0.005*"great" + 0.004*"well" + 0.004*"see"')
(23, '0.009*"upon" + 0.006*"like" + 0.006*"see" + 0.006*"mr" + 0.006*"man" + 0.006*"must" + 0.005*"may" + 0.005*"great" + 0.005*"eyes" + 0.005*"well"')
(29, '0.007*"upon" + 0.006*"little" + 0.006*"may" + 0.005*"see" + 0.005*"great" + 0.005*"day" + 0.005*"man" + 0.005*"mr" + 0.005*"never" + 0.005*"much"')
(5, '0.006*"man" + 0.006*"life" + 0.006*"like" + 0.006*"us" + 0.005*"may" + 0.005*"upon" + 0.005*"mr" + 0.005*"know" + 0.004*"must" + 0.004*"much"')
(3, '0.008*"upon" + 0.007*"like" + 0.007*"mr" + 0.006*"little" + 0.006*"see" + 0.006*"well" + 0.006*"know" + 0.005*"men" + 0.005*"good" + 0.005*"may"')
""":
        print(i, id2word[i])

In [5]:
print(len(corpus))
print(len(corpus[0]))

182791
398


In [9]:
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                       id2word=id2word,
                                       num_topics=30, random_state=RANDOM_SEED, workers=10) # workers=10

for topic in lda_model.print_topics():
    print(topic)

#lda_model.save("coha/lda_coha.model")


(26, '0.011*"mr" + 0.010*"upon" + 0.008*"mrs" + 0.005*"shall" + 0.005*"us" + 0.004*"think" + 0.004*"miss" + 0.004*"oh" + 0.003*"thought" + 0.003*"right"')
(28, '0.007*"upon" + 0.005*"face" + 0.004*"us" + 0.004*"away" + 0.004*"love" + 0.004*"right" + 0.004*"eyes" + 0.004*"mind" + 0.004*"mr" + 0.004*"young"')
(9, '0.007*"upon" + 0.006*"mr" + 0.005*"men" + 0.005*"life" + 0.004*"us" + 0.004*"shall" + 0.004*"house" + 0.004*"yet" + 0.004*"hand" + 0.003*"place"')
(12, '0.008*"upon" + 0.006*"men" + 0.006*"sir" + 0.004*"th" + 0.004*"us" + 0.004*"life" + 0.003*"place" + 0.003*"hand" + 0.003*"mr" + 0.003*"shall"')
(15, '0.006*"mr" + 0.005*"work" + 0.005*"men" + 0.004*"life" + 0.004*"dr" + 0.004*"upon" + 0.003*"case" + 0.003*"year" + 0.003*"though" + 0.003*"us"')
(1, '0.005*"mr" + 0.005*"life" + 0.005*"upon" + 0.005*"us" + 0.004*"men" + 0.004*"father" + 0.004*"young" + 0.004*"came" + 0.003*"yet" + 0.003*"heart"')
(18, '0.007*"upon" + 0.004*"us" + 0.004*"king" + 0.004*"life" + 0.004*"shall" + 0.004

In [7]:

with open("coha/lda_coha/lda_corpus_coha2.json", "w") as f:
    json.dump(corpus, f)

## Viualization and examination of topics
The code in the following cell is taken from this notebook [https://colab.research.google.com/github/littlecolumns/ds4j-notebooks/blob/master/text-analysis/notebooks/Topic%20models%20with%20Gensim.ipynb]. Also, I coulndn't get it to work.

In [10]:
import pyLDAvis
import pyLDAvis.gensim

lda_model = gensim.models.ldamodel.LdaModel.load("lda.model")
id2word =  gensim.corpora.dictionary.Dictionary.load("lda.model.id2word")
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
vis

: 